# RDDT ATTRv V4 pipeline

This notebook is a thin Snowflake-workspace runner. The corrected workbook is the sole clinical source of truth. It reads the warehouse tables in place and creates session temporary `AMY_V4_*` tables only.

In [ ]:
from pathlib import Path
import json
import sys

package_candidates = [Path.cwd() / 'v4', Path.cwd(), Path.cwd().parent / 'v4']
package_dir = next((p.resolve() for p in package_candidates if (p / '__init__.py').exists() and (p / 'config').is_dir()), None)
if package_dir is None:
    raise RuntimeError('Run this notebook from the v4 folder or its parent directory.')
project_root = package_dir.parent
sys.path.insert(0, str(project_root))

from snowflake.snowpark.context import get_active_session
import medspacy
from v4.config_loader import load_runtime_config
from v4.pipeline import TEMP_TABLES, run_attrv_v4_pipeline
from v4.warehouse.source_schema import default_source_config

config_dir = package_dir / 'config'
export_dir = Path.cwd() / 'rddt_v4_exports'
session = get_active_session()
config = load_runtime_config(config_dir, phenotype='ATTRV')
nlp = medspacy.load()
print({'config_hash': config.config_hash})

## Physical source location

Set `namespace` only when the configured tables are not resolved by the active database/schema. Do not add social-history or medication sources.

In [ ]:
source_config = default_source_config()
source_config['namespace'] = None  # Example: 'DATABASE.SCHEMA'
SCREENING_CUTOFF = None  # REQUIRED: set an as-of date, for example '2026-01-31'
if SCREENING_CUTOFF is None:
    raise ValueError('Set SCREENING_CUTOFF before running; post-cutoff evidence must not leak into screening.')

run = run_attrv_v4_pipeline(
    session,
    config_dir=config_dir,
    source_config=source_config,
    nlp=nlp,
    screening_cutoff=SCREENING_CUTOFF,
    persist_intermediates=True,
    include_profiles=True,
    profile_output_dir=export_dir,
)
print(json.dumps(run.summary(), indent=2, default=str))

## Review and download

The router table can be downloaded from the Snowflake result grid. The JSONL/CSV files are written to the notebook runtime. External exports omit proprietary pipeline trace by default.

In [ ]:
router_df = session.table(TEMP_TABLES['router_output'])
router_df.show()
print('Profile exports:', run.profile_exports)
try:
    from IPython.display import FileLink, display
    for path in run.profile_exports.values():
        display(FileLink(path))
except Exception:
    pass

## Audit-only outputs

Config gaps and the full internal trace are for authorized validation. Do not distribute `proprietary_pipeline_trace`; the exported files above exclude it by default.

In [ ]:
print(json.dumps(run.config_gaps[:50], indent=2, default=str))
if run.patient_profiles:
    audit_trace = run.patient_profiles[0].get('proprietary_pipeline_trace', {})
    print(json.dumps(audit_trace, indent=2, default=str)[:10000])